# Musanga V4 — NB04 v4_004 | RGB Model A final evaluation (cached results only)

**Purpose:** Report the selected RGB **Model A** on the same 30 exhaustive circles and 76 RGB-covered plots, with equal-unit, 50/50 circle–plot-balanced, and valid-pixel-weighted metrics; forest-level summaries; fractional-cover scatterplots; four largest outliers; cover-stratified analysis; and pooled pixel discrimination.

**Fast and safe:** Reuses the **106 saved held-out per-unit `.npz` evaluations from NB04 v4_003** (or its previously aggregated CSV); does **not** read mosaics, calculate chips, run inference, load model checkpoints, train, or modify old results. Existing `Outputs/v4/04_paired_rgb_heldout_v1/` remains read-only. New summaries are saved under `Outputs/v4/04_RGB_A_final_reporting_v1/`.

**Reference caution:** Plot background is treated as exhaustively annotated according to NB03's provisional assumption. Unit-mean segmentation metrics ignore undefined per-unit values; pixel-pooled F1/IoU are provided separately.

## 00 — Mount Drive and load saved evaluation (no inference)

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

ROOTS=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
       Path('/content/drive/MyDrive/Musanga/Final')]
BASE=next((r/'Outputs/v4/04_paired_rgb_heldout_v1' for r in ROOTS
           if (r/'Outputs/v4/04_paired_rgb_heldout_v1').exists()),None)
assert BASE is not None, 'NB04 saved output directory not found; check Drive mount'
OUT=BASE.parent/'04_RGB_A_final_reporting_v1'
OUT.mkdir(parents=True,exist_ok=True)

# Prefer the precomputed CSV, but reconstruct from original saved per-unit files
# if the CSV is missing. Both paths are read-only to original outputs.
METRICS_CSV=BASE/'04_heldout_unit_metrics.csv'
if METRICS_CSV.is_file():
    df=pd.read_csv(METRICS_CSV)
    assert 'arm' in df.columns
    df=df[df.arm.eq('A')].copy()
    source='saved aggregate CSV'
else:
    files=sorted((BASE/'per_unit').glob('fold*_*.npz'))
    print('Existing per-unit evaluation cache:',len(files),'files')
    records=[]
    for f in files:
        with np.load(f,allow_pickle=False) as z:
            meta=json.loads(str(z['metadata'].item()))
            data=json.loads(str(z['A'].item()))
            records.append({**meta,**data,'arm':'A'})
    df=pd.DataFrame(records)
    source='saved per-unit NPZ cache'

assert len(df)==106, f'Expected 106 held-out units, got {len(df)}. Missing caches must be completed with earlier NB04; no inference is attempted here.'
df['unit_id']=df.unit_id.astype(str)
df['fold']=df.fold.astype(int)
assert df.unit_type.value_counts().to_dict()=={'plot':76,'circle':30}
assert not df.duplicated(['unit_type','unit_id']).any()
assert set(df.fold)=={0,1,2}
cols=['observed_cover_pp','predicted_cover_pp','n_pixels','F1','IoU','precision','recall',
      'brier','tp','fp','tn','fn']
assert set(cols).issubset(df.columns),set(cols)-set(df.columns)
assert df[['observed_cover_pp','predicted_cover_pp','n_pixels']].notna().all().all()
assert df.n_pixels.gt(0).all()
df['cover_error_pp']=df.predicted_cover_pp-df.observed_cover_pp
df['abs_cover_error_pp']=df.cover_error_pp.abs()
df.to_csv(OUT/'04_RGB_A_106_unit_metrics.csv',index=False)
print('Loaded:',source,'| 30 circles + 76 plots | folds:',sorted(df.fold.unique()))
print('No mosaics read, no inference, no retraining.')
print('Reporting folder:',OUT)

## 01 — Weighted metrics: equal units, balanced circle–plot, evaluated-pixel weighting

In [ ]:
def wm(x,w):
    x=np.asarray(x,float);w=np.asarray(w,float)
    valid=np.isfinite(x)&np.isfinite(w)&(w>0)
    return float(np.average(x[valid],weights=w[valid])) if valid.any() else np.nan

def weighted_quantile(x,w,qs):
    x=np.asarray(x,float);w=np.asarray(w,float)
    valid=np.isfinite(x)&np.isfinite(w)&(w>0)
    x=x[valid];w=w[valid]
    if not len(x):return np.full(len(qs),np.nan)
    ix=np.argsort(x);x=x[ix];w=w[ix]
    c=(np.cumsum(w)-w/2)/w.sum()
    return np.interp(qs,c,x)

def weights(g,method):
    if method=='Equal units':return np.ones(len(g),float)
    if method=='Balanced circle/plot':
        n=g.unit_type.value_counts()
        return g.unit_type.map(lambda t:1/n[t]).to_numpy(float)
    if method=='Valid-pixel weighted':return g.n_pixels.to_numpy(float)
    raise ValueError(method)

def summarize(g,method):
    w=weights(g,method)
    obs=g.observed_cover_pp.to_numpy(float)
    pred=g.predicted_cover_pp.to_numpy(float)
    error=pred-obs
    mae=wm(np.abs(error),w)
    rmse=np.sqrt(wm(error**2,w))
    bias=wm(error,w)
    var=wm((obs-wm(obs,w))**2,w)
    sd=np.sqrt(var)
    q1,q3=weighted_quantile(obs,w,[.25,.75])
    # Weighted means of unit-level segmentation metrics (not pixel-pooled F1).
    result={'n':len(g),'F1':wm(g.F1,w),'IoU':wm(g.IoU,w),
            'Precision':wm(g.precision,w),'Recall':wm(g.recall,w),
            'R2':1-rmse**2/var if var>0 else np.nan,
            'MAE_pp':mae,'RMSE_pp':rmse,'Bias_pp':bias,
            'RPD':sd/rmse if rmse>0 else np.nan,
            'RPIQ':(q3-q1)/rmse if rmse>0 else np.nan,
            'Brier':wm(g.brier,w)}
    return result

SCHEMES=['Equal units','Balanced circle/plot','Valid-pixel weighted']
rows=[]
for group,g in [('ALL',df),*list(df.groupby('site',sort=True))]:
    for method in SCHEMES:
        rows.append({'Group':group,'Weighting':method,**summarize(g,method)})
W=pd.DataFrame(rows)
W.to_csv(OUT/'04_RGB_A_weighted_metrics.csv',index=False)
print('ALL: three weighting schemes')
display(W[W.Group.eq('ALL')].round(3))
print('PER FOREST: 50/50 reference-type balance within each forest')
display(W[W.Weighting.eq('Balanced circle/plot')].round(3))
print('Note: 50/50 type balancing is recomputed within each forest; F1/IoU are weighted unit means.')

## 02 — Separate reference-type diagnostics and pixel-pooled performance

In [ ]:
type_rows=[]
for label,g in [('ALL',df),*list(df.groupby('site',sort=True))]:
    for typ,t in g.groupby('unit_type'):
        type_rows.append({'Group':label,'unit_type':typ,**summarize(t,'Equal units')})
BY_TYPE=pd.DataFrame(type_rows)
BY_TYPE.to_csv(OUT/'04_RGB_A_by_site_and_reference_type.csv',index=False)
display(BY_TYPE.round(3))

# Micro metrics pool independent evaluation pixels through saved confusion counts.
micro=[]
for label,g in [('ALL',df),*list(df.groupby('site',sort=True))]:
    tp,fp,fn,tn=(int(g[k].sum()) for k in ['tp','fp','fn','tn'])
    micro.append(dict(Group=label,n_units=len(g),n_pixels=int(g.n_pixels.sum()),
                      F1_micro=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan,
                      IoU_micro=tp/(tp+fp+fn) if tp+fp+fn else np.nan,
                      Precision_micro=tp/(tp+fp) if tp+fp else np.nan,
                      Recall_micro=tp/(tp+fn) if tp+fn else np.nan,
                      tp=tp,fp=fp,fn=fn,tn=tn))
MICRO=pd.DataFrame(micro)
MICRO.to_csv(OUT/'04_RGB_A_pooled_pixel_metrics.csv',index=False)
print('Pixel-pooled segmentation metrics (different from the weighted unit means above):')
display(MICRO.round(3))

## 03 — Combined cover scatterplot and four largest outliers

In [ ]:
top4=df.nlargest(4,'abs_cover_error_pp')[
    ['site','unit_type','unit_id','fold','observed_cover_pp',
     'predicted_cover_pp','cover_error_pp','abs_cover_error_pp']].copy()
top4.to_csv(OUT/'04_RGB_A_top4_outliers.csv',index=False)
print('FOUR largest absolute cover errors')
display(top4.round(2))

colors={'circle':'#2477ad','plot':'#db8742'}
markers={'circle':'o','plot':'s'}
fig,axs=plt.subplots(1,2,figsize=(13,6),constrained_layout=True)
for i,ax in enumerate(axs):
    for typ,g in df.groupby('unit_type'):
        sizes=np.full(len(g),42.) if i==0 else np.clip(
            20+25*np.sqrt(g.n_pixels.to_numpy()/df.n_pixels.median()),18,220)
        ax.scatter(g.observed_cover_pp,g.predicted_cover_pp,
                   s=sizes,c=colors[typ],marker=markers[typ],
                   alpha=.68,edgecolors='white',linewidth=.4,
                   label=f'{typ.capitalize()} (n={len(g)})')
    ax.plot([0,100],[0,100],'k--',lw=1)
    ax.set(xlim=(-2,102),ylim=(-2,102),
           xlabel='Reference Musanga cover (%)',
           ylabel='Predicted Musanga cover (%)',
           title='Equal marker sizes' if i==0 else 'Marker sizes reflect evaluated pixels')
    ax.set_aspect('equal',adjustable='box');ax.grid(alpha=.15);ax.legend(frameon=False)
    for z in top4.itertuples():
        ax.scatter(z.observed_cover_pp,z.predicted_cover_pp,
                   s=155,facecolors='none',edgecolors='#b52c30',linewidth=1.8,zorder=5)
        ax.annotate(f'{z.site}: {z.unit_id}',
                    (z.observed_cover_pp,z.predicted_cover_pp),xytext=(5,6),
                    textcoords='offset points',fontsize=8,weight='bold')
fig.suptitle('RGB Model A | 30 circles + 76 plots | spatially held-out',fontsize=13)
fig.savefig(OUT/'04_RGB_A_cover_scatter_top4.png',dpi=220,bbox_inches='tight')
plt.show()

## 04 — Cover strata, forest plots and final report

In [ ]:
bins=[-.001,1,5,20,40,100.001]
labels=['0–1%','1–5%','5–20%','20–40%','>40%']
df['cover_stratum']=pd.cut(df.observed_cover_pp,bins=bins,
                           labels=labels,include_lowest=True)
strata=[]
for (typ,stratum),g in df.groupby(['unit_type','cover_stratum'],observed=True):
    strata.append({'unit_type':typ,'cover_stratum':str(stratum),
                   **summarize(g,'Equal units')})
STRATA=pd.DataFrame(strata)
STRATA.to_csv(OUT/'04_RGB_A_cover_strata.csv',index=False)
print('Cover-stratified diagnostics:')
display(STRATA[['unit_type','cover_stratum','n','F1','IoU','MAE_pp','Bias_pp']].round(3))

fig,axs=plt.subplots(2,2,figsize=(11,10),constrained_layout=True)
for ax,site in zip(axs.flat,sorted(df.site.unique())):
    g=df[df.site.eq(site)]
    for typ,t in g.groupby('unit_type'):
        ax.scatter(t.observed_cover_pp,t.predicted_cover_pp,s=36,
                   color=colors[typ],marker=markers[typ],alpha=.75,label=typ)
    ax.plot([0,100],[0,100],'k--',lw=1)
    ax.set(xlim=(-2,102),ylim=(-2,102),title=f'{site} (n={len(g)})',
           xlabel='Reference cover (%)',ylabel='Predicted cover (%)')
    ax.grid(alpha=.15);ax.legend(frameon=False)
fig.savefig(OUT/'04_RGB_A_scatter_by_forest.png',dpi=200,bbox_inches='tight')
plt.show()

manifest={'notebook':'NB04_v4_004_RGB_A_final','model':'A',
          'reference_counts':{'circle':30,'plot':76},
          'source_folder':str(BASE),'out_folder':str(OUT),
          'source':source,'inference_recomputed':False,
          'weighting':['equal_unit','equal_reference_type','evaluated_valid_pixel'],
          'exhaustive_plot_background_assumption':'provisional, inherited from NB03'}
(OUT/'04_RGB_A_reporting_manifest.json').write_text(json.dumps(manifest,indent=2))
print('COMPLETE — all Model A reporting saved to:',OUT)

## Interpretation

Use **balanced circle/plot** as the principal combined unit-level summary; retain equal-unit and valid-pixel-weighted sensitivity analyses. The **pixel-pooled** F1/IoU are a different estimand from macro F1/IoU. The four highlighted outliers are ranked by absolute cover error (pp), not regression residuals. No additional inference is performed.